# Precision test on the RTX 5090: bf16 vs fp8, same machine

**Upload both files into `work`:** this notebook and `stage2_test_data.json`. Then run the cells in order (cell 2 asks for your Hugging Face token).

Question: does serving precision alone change keyword invention? In the paper, precision moved LLaMA 3.1-8B by 0.194, but that
comparison also changed provider. Here everything is held fixed except precision: same GPU, same vLLM, same prompts, same seed.

Runs, in priority order (each saved as soon as it finishes):
1. LLaMA 3.1-8B **bf16**
2. LLaMA 3.1-8B **fp8**
3. Qwen 2.5-7B **bf16**
4. Qwen 2.5-7B **fp8**

Per run, two conditions on all 50 queries, both platforms: `arm2` (no schema) and `pv` (flat schema, prompt-and-validate).
Stage 1 is run once per model at bf16 and reused at fp8, so only Stage 2 precision differs.

Time: each model downloads once (about 20 minutes each); fp8 reuses the same weights. The calls themselves take a few minutes per run.
**Pilot on this server only. Do not mix with the OpenRouter numbers.** Download `precision_*` files before the session ends.

In [7]:
!pip install -U vllm
!pip uninstall -y torchaudio
!which vllm
!vllm --version

  Using cached nvidia_nvshmem_cu13-3.4.5-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (2.1 kB)
  Using cached nvidia_cuda_nvrtc-13.0.88-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_runtime-13.0.96-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cufft-12.0.0.61-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_cufile-1.15.1.6-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_cupti-13.0.85-py3-none-manylinux_2_25_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_curand-10.4.0.35-py3-none-manylinux_2_27_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cusolver-12.0.4.66-py3-none-manylinux_2_27_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_cusparse-12.6.3.3-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_nvtx-13.0.85-

In [1]:
# 1. Settings
N_QUERIES = 50
RUNS = [("llama", "meta-llama/Llama-3.1-8B-Instruct", "bf16"),
        ("llama", "meta-llama/Llama-3.1-8B-Instruct", "fp8"),
        ("qwen",  "Qwen/Qwen2.5-7B-Instruct",         "bf16"),
        ("qwen",  "Qwen/Qwen2.5-7B-Instruct",         "fp8")]
CONDITIONS = ["arm2", "pv"]
CONCURRENCY = 8
GPU_UTIL = "0.85"           # the GPU is shared; leave room for the other user's memory
MAX_LEN = "8192"            # real calls peaked at 5,142 tokens in the 4070 Ti pilot

import os, sys, subprocess, time, json, shutil, signal, re, csv, random, urllib.request, urllib.error
from concurrent.futures import ThreadPoolExecutor
from collections import defaultdict

HERE = os.getcwd()
WORK = os.path.expanduser("~/work"); os.makedirs(WORK, exist_ok=True); os.chdir(WORK)
PORT = 8000
VLLM = shutil.which("vllm") or "/opt/conda/bin/vllm"
PY = os.path.join(os.path.dirname(VLLM), "python")
_found = [c for c in (os.path.join(d, "stage2_test_data.json") for d in (HERE, WORK, os.path.expanduser("~"))) if os.path.exists(c)]
if not _found:
    raise SystemExit("stage2_test_data.json not found. Upload it next to this notebook.")
DATA = json.load(open(_found[0], encoding="utf-8"))
QUERIES = DATA["queries"][:N_QUERIES]
PLATFORMS = ["coursera", "edx"]
ELIGIBLE = {p: set(DATA["vocab"][p]["eligible"]) for p in PLATFORMS}
WINDOW = {p: DATA["vocab"][p]["window"] for p in PLATFORMS}
MAX_KW = DATA["max_keywords"]
BUDGET = 128 + 32 * MAX_KW
print(len(QUERIES), "queries | budget", BUDGET, "tokens | data:", _found[0])

50 queries | budget 1408 tokens | data: /home/jovyan/work/TA - Sean Matthew/stage2_test_data.json


In [2]:
# 2. Hugging Face token (LLaMA is gated). Paste a READ token; it is kept in memory only, never saved.
from getpass import getpass
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("HF read token: ").strip()
r = subprocess.run([PY, "-m", "pip", "uninstall", "-y", "torchaudio"], capture_output=True, text=True)
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip()
print(smi)
used, total = [int(x) for x in smi.split(",")[1:3]]
if total - used < 0.85 * total - 500:
    print(f"WARNING: someone else is using {used} MiB. If the server fails to start, lower GPU_UTIL in cell 1.")

HF read token:  ········


NVIDIA GeForce RTX 5090, 3087, 32607


In [3]:
# 3. Server, request and scoring helpers
def healthy():
    try:
        with urllib.request.urlopen(f"http://localhost:{PORT}/health", timeout=2) as r:
            return r.status == 200
    except Exception:
        return False

def stop_server(p):
    if p is None:
        return
    try:
        os.killpg(os.getpgid(p.pid), signal.SIGTERM); p.wait(timeout=60)
    except Exception:
        try: os.killpg(os.getpgid(p.pid), signal.SIGKILL)
        except Exception: pass
    time.sleep(5)

def start_server(model, precision, tag):
    args = [VLLM, "serve", model, "--port", str(PORT), "--dtype", "bfloat16", "--max-model-len", MAX_LEN,
            "--gpu-memory-utilization", GPU_UTIL, "--generation-config", "vllm"]
    if precision == "fp8":
        args += ["--quantization", "fp8"]
    log = open(os.path.join(WORK, f"precision_server_{tag}.log"), "w")
    p = subprocess.Popen(args, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    t0 = time.time()
    while not healthy():
        if p.poll() is not None:
            raise SystemExit(f"Server failed; paste the last lines of precision_server_{tag}.log to Claude.")
        if time.time() - t0 > 60 * 60:
            stop_server(p); raise SystemExit("Server start timed out.")
        time.sleep(5)
    print(f"  server ready after {(time.time() - t0) / 60:.1f} min")
    return p

def chat(model, prompt, max_tokens):
    body = {"model": model, "messages": [{"role": "user", "content": prompt}],
            "temperature": 0, "seed": 34, "max_tokens": max_tokens}
    req = urllib.request.Request(f"http://localhost:{PORT}/v1/chat/completions",
                                 json.dumps(body).encode(), {"Content-Type": "application/json"})
    t = time.time()
    try:
        r = json.load(urllib.request.urlopen(req, timeout=900))
    except urllib.error.HTTPError as e:
        return {"error": f"HTTP {e.code}: {e.read().decode(errors='replace')[:400]}", "seconds": time.time() - t}
    except Exception as e:
        return {"error": f"{type(e).__name__}: {e}", "seconds": time.time() - t}
    ch = r["choices"][0]
    return {"text": ch["message"]["content"] or "", "finish": ch.get("finish_reason"),
            "completion_tokens": r["usage"]["completion_tokens"], "prompt_tokens": r["usage"]["prompt_tokens"],
            "seconds": time.time() - t}

def first_json(text):
    t = re.sub(r"^\s*```(?:json)?|```\s*$", "", text.strip(), flags=re.M).strip()
    a, b = t.find("{"), t.rfind("}")
    if a < 0 or b <= a:
        return None
    try:
        return json.loads(t[a:b + 1])
    except Exception:
        return None

def schema_problem(obj):
    if not isinstance(obj, dict): return "not an object"
    if set(obj) != {"platform", "keywords"}: return f"top-level keys {sorted(obj)}"
    if obj["platform"] not in PLATFORMS: return "bad platform"
    kw = obj["keywords"]
    if not isinstance(kw, dict): return "keywords not an object"
    if not 1 <= len(kw) <= MAX_KW: return f"{len(kw)} keywords (allowed 1 to {MAX_KW})"
    for v in kw.values():
        if isinstance(v, bool) or not isinstance(v, (int, float)) or not 0 <= v <= 1:
            return "score out of range"
    return None

def build(cond, platform, query):
    vocab = ", ".join(WINDOW[platform])
    if cond == "arm2":
        return DATA["prompt_arm2"].format(platform=platform, vocabulary=vocab, query=query, max_keywords=MAX_KW)
    return DATA["prompt_arm3_v5"].format(platform=platform, vocabulary=vocab, query=query,
                                         max_keywords=MAX_KW, schema=DATA["schema_v5_rendered"])

def keywords_from(cond, r):
    if "error" in r: return [], "request_error"
    if r["finish"] == "length": return [], "overrun"
    obj = first_json(r["text"])
    if cond == "arm2":
        if isinstance(obj, dict):
            kw = obj.get("keywords") if isinstance(obj.get("keywords"), dict) else obj
            return [str(k) for k in kw][:MAX_KW], "ok"
        return [], "unparseable"
    if obj is None: return [], "violation: unparseable"
    prob = schema_problem(obj)
    if prob: return [], f"violation: {prob}"
    return [str(k) for k in obj["keywords"]], "ok"

print("ok")

ok


In [4]:
# 4. Run everything (each run is saved as soon as it finishes; re-running skips finished runs)
rows_path = os.path.join(WORK, "precision_results.csv")
ROWS = list(csv.DictReader(open(rows_path, encoding="utf-8"))) if os.path.exists(rows_path) else []
for r in ROWS:
    r["n_keywords"], r["n_invalid"] = int(r["n_keywords"]), int(r["n_invalid"])
done = {(r["family"], r["precision"]) for r in ROWS}

def save():
    with open(rows_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(ROWS[0])); w.writeheader(); w.writerows(ROWS)

server = None
for family, model, precision in RUNS:
    if (family, precision) in done:
        print(f"{family} {precision}: already done, skipping"); continue
    tag = f"{family}_{precision}"
    print(f"{tag}: starting server (first start of a model downloads about 16 GB)")
    t_run = time.time()
    server = start_server(model, precision, tag)
    try:
        exp_path = os.path.join(WORK, f"precision_expanded_{family}.json")
        if os.path.exists(exp_path):
            EXPANDED = json.load(open(exp_path))
        else:
            def expand(q):
                r = chat(model, DATA["expand_prompt"].format(query=q["query"]), 512)
                return q["id"], (r.get("text") or "").strip() or q["query"]
            with ThreadPoolExecutor(CONCURRENCY) as ex:
                EXPANDED = dict(ex.map(expand, QUERIES))
            json.dump(EXPANDED, open(exp_path, "w"), indent=1)
            print(f"  Stage 1 done at {precision}; reused for this model's other precision")

        def run_one(args):
            cond, q, platform = args
            r = chat(model, build(cond, platform, EXPANDED[q["id"]]), BUDGET)
            kws, outcome = keywords_from(cond, r)
            invalid = [k for k in kws if k.strip().lower() not in ELIGIBLE[platform]]
            return {"family": family, "precision": precision, "condition": cond, "query_id": q["id"],
                    "platform": platform, "outcome": outcome, "n_keywords": len(kws), "n_invalid": len(invalid),
                    "invalid_examples": "; ".join(invalid[:5]), "finish": r.get("finish"),
                    "prompt_tokens": r.get("prompt_tokens"), "completion_tokens": r.get("completion_tokens"),
                    "seconds": round(r.get("seconds", 0), 1), "error": r.get("error", ""),
                    "raw": (r.get("text") or "")[:3000]}

        for cond in CONDITIONS:
            t = time.time()
            with ThreadPoolExecutor(CONCURRENCY) as ex:
                out = list(ex.map(run_one, [(cond, q, p) for q in QUERIES for p in PLATFORMS]))
            ROWS += out; save()
            errs = [r["error"] for r in out if r["error"]]
            print(f"  {cond}: {len(out)} calls in {(time.time() - t) / 60:.1f} min, "
                  f"{sum(r['outcome'] != 'ok' for r in out)} not ok" + (f" | first error: {errs[0][:200]}" if errs else ""))
    finally:
        stop_server(server); server = None
    print(f"{tag}: finished in {(time.time() - t_run) / 60:.1f} min, saved")

llama_bf16: starting server (first start of a model downloads about 16 GB)
  server ready after 22.8 min
  Stage 1 done at bf16; reused for this model's other precision
  arm2: 100 calls in 2.0 min, 16 not ok
  pv: 100 calls in 1.5 min, 28 not ok
llama_bf16: finished in 26.5 min, saved
llama_fp8: starting server (first start of a model downloads about 16 GB)
  server ready after 1.0 min
  arm2: 100 calls in 1.4 min, 18 not ok
  pv: 100 calls in 0.8 min, 23 not ok
llama_fp8: finished in 3.4 min, saved
qwen_bf16: starting server (first start of a model downloads about 16 GB)
  server ready after 20.5 min
  Stage 1 done at bf16; reused for this model's other precision
  arm2: 100 calls in 0.4 min, 0 not ok
  pv: 100 calls in 0.9 min, 11 not ok
qwen_bf16: finished in 22.0 min, saved
qwen_fp8: starting server (first start of a model downloads about 16 GB)
  server ready after 0.9 min
  arm2: 100 calls in 0.3 min, 1 not ok
  pv: 100 calls in 0.5 min, 11 not ok
qwen_fp8: finished in 1.9 min, 

In [5]:
# 5. Results: per query, both platforms pooled (as in the paper), plus the paired bf16 - fp8 difference
def per_query(family, precision, cond):
    byq = defaultdict(list)
    for r in ROWS:
        if (r["family"], r["precision"], r["condition"]) == (family, precision, cond):
            byq[r["query_id"]].append(r)
    itt, at = {}, {}
    for q, v in byq.items():
        if any(x["outcome"] == "request_error" for x in v):
            continue
        n = sum(int(x["n_keywords"]) for x in v); bad = sum(int(x["n_invalid"]) for x in v)
        itt[q] = 1.0 if n == 0 else bad / n
        if n: at[q] = bad / n
    return itt, at, byq

def mean(d): return round(sum(d.values()) / len(d), 4) if d else None

lines = ["| model | precision | condition | queries | ITT invalid | as-treated invalid | silent | violations | overruns | keywords/call |",
         "|---|---|---|---|---|---|---|---|---|---|"]
for family, _, precision in RUNS:
    for cond in CONDITIONS:
        itt, at, byq = per_query(family, precision, cond)
        if not byq: continue
        calls = [x for v in byq.values() for x in v if x["outcome"] != "request_error"]
        lines.append(f"| {family} | {precision} | {cond} | {len(itt)} | {mean(itt)} | {mean(at)} | {len(itt) - len(at)} | "
                     f"{sum(x['outcome'].startswith('violation') for x in calls)} | {sum(x['outcome'] == 'overrun' for x in calls)} | "
                     f"{round(sum(int(x['n_keywords']) for x in calls) / max(len(calls), 1), 1)} |")

def boot_ci(diffs, n=10000, seed=34):
    rnd = random.Random(seed); k = len(diffs)
    ms = sorted(sum(rnd.choice(diffs) for _ in range(k)) / k for _ in range(n))
    return round(ms[int(0.025 * n)], 4), round(ms[int(0.975 * n)], 4)

lines += ["", "Paired difference, bf16 minus fp8 (ITT, same queries; 95% CI by bootstrapping queries):", ""]
for family in dict.fromkeys(f for f, _, _ in RUNS):
    for cond in CONDITIONS:
        a, _, _ = per_query(family, "bf16", cond); b, _, _ = per_query(family, "fp8", cond)
        qs = sorted(set(a) & set(b))
        if not qs: continue
        d = [a[q] - b[q] for q in qs]
        lines.append(f"- {family} {cond}: {round(sum(d) / len(d), 4)} over {len(qs)} queries, 95% CI {boot_ci(d)}")

note = ("\n\nRTX 5090, local vLLM, max context 8,192, greedy, seed 34, --generation-config vllm, "
        "fp8 = vLLM online quantization of the bf16 weights. One pass. Not comparable with the OpenRouter results.\n")
text = "\n".join(lines) + note
open(os.path.join(WORK, "precision_summary.md"), "w").write(text)
print(text)
print("download:", [f for f in sorted(os.listdir(WORK)) if f.startswith("precision_")])

| model | precision | condition | queries | ITT invalid | as-treated invalid | silent | violations | overruns | keywords/call |
|---|---|---|---|---|---|---|---|---|---|
| llama | bf16 | arm2 | 50 | 0.5368 | 0.5073 | 3 | 0 | 16 | 31.7 |
| llama | bf16 | pv | 50 | 0.5779 | 0.5411 | 4 | 17 | 11 | 22.7 |
| llama | fp8 | arm2 | 50 | 0.5665 | 0.5388 | 3 | 0 | 18 | 31.9 |
| llama | fp8 | pv | 50 | 0.5819 | 0.5355 | 5 | 15 | 8 | 23.3 |
| qwen | bf16 | arm2 | 50 | 0.4937 | 0.4937 | 0 | 0 | 0 | 16.2 |
| qwen | bf16 | pv | 50 | 0.5795 | 0.5795 | 0 | 6 | 5 | 18.1 |
| qwen | fp8 | arm2 | 50 | 0.4941 | 0.4941 | 0 | 0 | 1 | 16.3 |
| qwen | fp8 | pv | 50 | 0.5162 | 0.5162 | 0 | 4 | 7 | 18.3 |

Paired difference, bf16 minus fp8 (ITT, same queries; 95% CI by bootstrapping queries):

- llama arm2: -0.0297 over 50 queries, 95% CI (-0.0571, -0.003)
- llama pv: -0.0041 over 50 queries, 95% CI (-0.0663, 0.0623)
- qwen arm2: -0.0003 over 50 queries, 95% CI (-0.0393, 0.0369)
- qwen pv: 0.0633 over 50 queries,

## Emergency stop

In [ ]:
subprocess.run("pkill -f 'vllm serve'", shell=True); time.sleep(5)
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)